# 03b - Strong Models
Goal: train Random Forest and LightGBM for binary and multi-class detection, handle class imbalance, tune hyperparameters and save the best multi-class model.

## 1. Setup
Load the processed train/test sets and split them into features (X) and labels (y). `FEATURE_COLS` from `src/config.py` fixes the column order, so the saved model gets the same input order in the demo and the Mininet test.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.config import RANDOM_STATE, FEATURE_COLS, LABEL_COLS

TRAIN_PATH = '../data/processed/train.parquet'
TEST_PATH  = '../data/processed/test.parquet'

print('RANDOM_STATE:', RANDOM_STATE)
print('Number of features:', len(FEATURE_COLS))
print('Label columns:', LABEL_COLS)

RANDOM_STATE: 42
Number of features: 62
Label columns: ['label_multi', 'label_binary']


In [2]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

print('Train:', train.shape)
print('Test: ', test.shape)

Train: (341633, 64)
Test:  (85409, 64)


In [3]:
X_train = train[FEATURE_COLS]
X_test  = test[FEATURE_COLS]

y_train_bin = train['label_binary']
y_test_bin  = test['label_binary']

y_train_multi = train['label_multi']
y_test_multi  = test['label_multi']

print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train: (341633, 62) | X_test: (85409, 62)


Class distribution in train and test. Thanks to the stratified split, the percentages should be almost the same in both sets.

In [4]:
dist = pd.DataFrame({
    'train_count': y_train_multi.value_counts(),
    'train_%': (y_train_multi.value_counts(normalize=True) * 100).round(2),
    'test_count': y_test_multi.value_counts(),
    'test_%': (y_test_multi.value_counts(normalize=True) * 100).round(2),
})
dist

,train_count,train_%,test_count,test_%
label_multi,,,,
Benign,186996,54.74,46749,54.74
Dos,101205,29.62,25301,29.62
DDoS,40215,11.77,10054,11.77
Bot,7165,2.10,1792,2.10
Prob,6052,1.77,1513,1.77


In [5]:
print('Binary labels (0 = Benign, 1 = Attack), train %:')
print((y_train_bin.value_counts(normalize=True) * 100).round(2))

Binary labels (0 = Benign, 1 = Attack), train %:
label_binary
0    54.74
1    45.26
Name: proportion, dtype: float64


## 2. evaluate()
Copied unchanged from `notebooks/03_models.ipynb` (baseline-models branch, commit e343c28), so all models are measured in the same way.

It trains the model, measures training time and prediction time per flow (ms), prints the classification report and returns one row for the results table. The function always returns `imbalance='none'` and empty `notes`, so we set these fields on the returned row when needed. It does not return the predictions, so for confusion matrices we call `model.predict` again on the already trained model.

In [6]:
import time
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [7]:
def evaluate(model, name, task, X_tr, y_tr, X_te, y_te):
    # 1) train and measure time
    t0 = time.perf_counter()
    model.fit(X_tr,y_tr)
    train_time = time.perf_counter() - t0

    # 2) predict and measure time
    t0 = time.perf_counter()
    y_pred = model.predict(X_te)
    predict_time = time.perf_counter() - t0
    predict_ms = predict_time / len(X_te) * 1000 #converts seconds to ms

    # 3) show the detailed report
    print(f"=== {name} | {task} ===")
    print(classification_report(y_te,y_pred,digits=4))

    # 4) return one row for results.csv
    return{
        "model":name,
        "task":task,
        "imbalance":"none",
        "macro_f1":f1_score(y_te,y_pred,average="macro"),
        "accuracy":accuracy_score(y_te,y_pred),
        "train_time_s":round(train_time,2),
        "predict_ms_per_flow":round(predict_ms,5),
        "notes":"",

    }


In [8]:
results = []  # one dict per model, written to reports/results_strong.csv at the end

## 3. Random Forest
Random Forest trains many decision trees, each on a bootstrap sample of the rows and a random subset of features at each split, and takes a majority vote. Averaging many different trees reduces overfitting compared to a single tree.

No scaler: a tree splits on thresholds such as `Flow Duration > 1000`, so the scale of a feature does not change which split is chosen. Default hyperparameters are used here (100 trees); tuning comes in section 6. `n_jobs=-1` uses all CPU cores.

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
rf_bin = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

row = evaluate(rf_bin, "RandomForest", "binary",
               X_train, y_train_bin, X_test, y_test_bin)
results.append(row)
row

=== RandomForest | binary ===
              precision    recall  f1-score   support

           0     0.9213    0.9199    0.9206     46749
           1     0.9033    0.9049    0.9041     38660

    accuracy                         0.9131     85409
   macro avg     0.9123    0.9124    0.9124     85409
weighted avg     0.9131    0.9131    0.9131     85409



{'model': 'RandomForest',
 'task': 'binary',
 'imbalance': 'none',
 'macro_f1': 0.9123512667967151,
 'accuracy': 0.9131239096582328,
 'train_time_s': 41.1,
 'predict_ms_per_flow': 0.0064,
 'notes': ''}

In [11]:
rf_multi = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

row = evaluate(rf_multi, "RandomForest", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
results.append(row)
row

=== RandomForest | multi ===


              precision    recall  f1-score   support

      Benign     0.9195    0.9212    0.9203     46749
         Bot     0.2926    0.2896    0.2911      1792
        DDoS     0.5634    0.5651    0.5643     10054
         Dos     0.9106    0.9096    0.9101     25301
        Prob     0.5521    0.5254    0.5384      1513

    accuracy                         0.8556     85409
   macro avg     0.6476    0.6422    0.6448     85409
weighted avg     0.8553    0.8556    0.8554     85409



{'model': 'RandomForest',
 'task': 'multi',
 'imbalance': 'none',
 'macro_f1': 0.6448476159595093,
 'accuracy': 0.8555889894507605,
 'train_time_s': 41.34,
 'predict_ms_per_flow': 0.00948,
 'notes': ''}

## 4. LightGBM
LightGBM is a gradient boosting model. Unlike Random Forest, where trees are trained independently and vote, boosting trains small trees one after another and each new tree tries to correct the errors of the trees before it. LightGBM is fast because it bins feature values into histograms and grows each tree leaf by leaf.

Same setup as Random Forest for a fair comparison: default hyperparameters and no scaler (also tree-based). `verbose=-1` only hides LightGBM's training log.

In [12]:
from lightgbm import LGBMClassifier

In [13]:
lgbm_bin = LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

row = evaluate(lgbm_bin, "LightGBM", "binary",
               X_train, y_train_bin, X_test, y_test_bin)
results.append(row)
row

=== LightGBM | binary ===
              precision    recall  f1-score   support

           0     0.9817    0.8932    0.9353     46749
           1     0.8835    0.9798    0.9292     38660

    accuracy                         0.9324     85409
   macro avg     0.9326    0.9365    0.9322     85409
weighted avg     0.9372    0.9324    0.9325     85409



{'model': 'LightGBM',
 'task': 'binary',
 'imbalance': 'none',
 'macro_f1': 0.9322447775503686,
 'accuracy': 0.9323841749698509,
 'train_time_s': 8.2,
 'predict_ms_per_flow': 0.00208,
 'notes': ''}

In [14]:
lgbm_multi = LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

row = evaluate(lgbm_multi, "LightGBM", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
results.append(row)
row

=== LightGBM | multi ===


              precision    recall  f1-score   support

      Benign     0.9468    0.9238    0.9352     46749
         Bot     0.4381    0.0552    0.0981      1792
        DDoS     0.5717    0.8355    0.6788     10054
         Dos     0.9585    0.9074    0.9322     25301
        Prob     0.7798    0.4752    0.5906      1513

    accuracy                         0.8824     85409
   macro avg     0.7390    0.6394    0.6470     85409
weighted avg     0.8925    0.8824    0.8805     85409



{'model': 'LightGBM',
 'task': 'multi',
 'imbalance': 'none',
 'macro_f1': 0.6469781637595247,
 'accuracy': 0.8823777353674671,
 'train_time_s': 28.26,
 'predict_ms_per_flow': 0.00711,
 'notes': ''}

## 5. Class imbalance
Bot (2.1%) and Prob (1.8%) are rare. In section 4, LightGBM predicted 86% of Bot flows as DDoS: the two classes look similar and DDoS has 5.6 times more training rows, so the majority class is the safer guess.

`class_weight="balanced"` gives each class a weight of `n_samples / (n_classes * n_samples_in_class)`, so mistakes on rare classes cost more during training. It does not add or remove rows and is applied to the training data only.

The `none` results are already in sections 3 and 4. Here we train the `balanced` versions of both models for both tasks. `evaluate()` always returns `imbalance='none'`, so we set the field on the returned row.

In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_multi)
weights = compute_class_weight("balanced", classes=classes, y=y_train_multi)
pd.Series(weights, index=classes).round(2).sort_values()

Benign     0.37
Dos        0.68
DDoS       1.70
Bot        9.54
Prob      11.29
dtype: float64

In [16]:
balanced_models = {
    ("RandomForest", "binary"): RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    ("RandomForest", "multi"):  RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    ("LightGBM", "binary"):     LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    ("LightGBM", "multi"):      LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

for (name, task), model in balanced_models.items():
    y_tr, y_te = (y_train_bin, y_test_bin) if task == "binary" else (y_train_multi, y_test_multi)
    row = evaluate(model, name, task, X_train, y_tr, X_test, y_te)
    row["imbalance"] = "balanced"
    results.append(row)

=== RandomForest | binary ===
              precision    recall  f1-score   support

           0     0.9214    0.9198    0.9206     46749
           1     0.9032    0.9052    0.9042     38660

    accuracy                         0.9132     85409
   macro avg     0.9123    0.9125    0.9124     85409
weighted avg     0.9132    0.9132    0.9132     85409



=== RandomForest | multi ===


              precision    recall  f1-score   support

      Benign     0.9192    0.9213    0.9203     46749
         Bot     0.2903    0.2907    0.2905      1792
        DDoS     0.5618    0.5636    0.5627     10054
         Dos     0.9110    0.9092    0.9101     25301
        Prob     0.5481    0.5155    0.5313      1513

    accuracy                         0.8552     85409
   macro avg     0.6461    0.6401    0.6430     85409
weighted avg     0.8550    0.8552    0.8551     85409



=== LightGBM | binary ===
              precision    recall  f1-score   support

           0     0.9856    0.8888    0.9347     46749
           1     0.8798    0.9843    0.9291     38660

    accuracy                         0.9320     85409
   macro avg     0.9327    0.9365    0.9319     85409
weighted avg     0.9377    0.9320    0.9322     85409



=== LightGBM | multi ===


              precision    recall  f1-score   support

      Benign     0.9943    0.8734    0.9299     46749
         Bot     0.1900    0.8705    0.3119      1792
        DDoS     0.5100    0.4866    0.4980     10054
         Dos     0.9902    0.8871    0.9358     25301
        Prob     0.2771    0.7092    0.3985      1513

    accuracy                         0.8290     85409
   macro avg     0.5923    0.7654    0.6148     85409
weighted avg     0.9065    0.8290    0.8585     85409



Side-by-side comparison of `none` and `balanced` (macro F1 on the test set).

In [17]:
cmp = pd.DataFrame(results).pivot_table(index=["model", "task"], columns="imbalance", values="macro_f1")
cmp["diff"] = cmp["balanced"] - cmp["none"]
cmp.round(4)

imbalance            balanced    none    diff
model        task                            
LightGBM     binary    0.9319  0.9322 -0.0003
             multi     0.6148  0.6470 -0.0321
RandomForest binary    0.9124  0.9124  0.0000
             multi     0.6430  0.6448 -0.0019

Recall per class for the multi-class models: does `balanced` help Bot and Prob?

In [18]:
from sklearn.metrics import recall_score

multi_models = {
    ("RandomForest", "none"): rf_multi,
    ("RandomForest", "balanced"): balanced_models[("RandomForest", "multi")],
    ("LightGBM", "none"): lgbm_multi,
    ("LightGBM", "balanced"): balanced_models[("LightGBM", "multi")],
}

recall_table = pd.DataFrame({
    key: recall_score(y_test_multi, model.predict(X_test), labels=classes, average=None)
    for key, model in multi_models.items()
}, index=classes).T
recall_table.round(3)

Benign    Bot   DDoS    Dos   Prob
RandomForest none       0.921  0.290  0.565  0.910  0.525
             balanced   0.921  0.291  0.564  0.909  0.516
LightGBM     none       0.924  0.055  0.835  0.907  0.475
             balanced   0.873  0.871  0.487  0.887  0.709

### SMOTE (LightGBM, multi-class)
SMOTE creates synthetic rows for the minority classes by interpolating between a sample and one of its nearest neighbours of the same class. With the default settings every class is oversampled to the size of the largest class (Benign).

SMOTE is put inside an `imblearn.pipeline.Pipeline`, so it runs only during `fit` on the training data. During `predict` the step is skipped and the test set is used as it is.

We only try it for LightGBM multi-class: `balanced` had no effect on Random Forest and on the binary task, where the classes are already close to balanced.

Limitation: SMOTE finds neighbours with Euclidean distance on unscaled features, so large-valued features such as `Flow Duration` dominate the neighbour search.

In [19]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

lgbm_smote = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
])

row = evaluate(lgbm_smote, "LightGBM", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
row["imbalance"] = "smote"
results.append(row)
row

=== LightGBM | multi ===


              precision    recall  f1-score   support

      Benign     0.9940    0.8744    0.9304     46749
         Bot     0.1952    0.8594    0.3181      1792
        DDoS     0.5153    0.5148    0.5151     10054
         Dos     0.9894    0.8864    0.9351     25301
        Prob     0.2911    0.7085    0.4127      1513

    accuracy                         0.8324     85409
   macro avg     0.5970    0.7687    0.6223     85409
weighted avg     0.9071    0.8324    0.8609     85409



{'model': 'LightGBM',
 'task': 'multi',
 'imbalance': 'smote',
 'macro_f1': 0.6222580815776257,
 'accuracy': 0.8323712957650833,
 'train_time_s': 107.51,
 'predict_ms_per_flow': 0.009,
 'notes': ''}

Comparison of the three imbalance settings for LightGBM multi-class.

In [20]:
multi_models[("LightGBM", "smote")] = lgbm_smote

lgbm_rows = pd.DataFrame(results).query("model == 'LightGBM' and task == 'multi'")
lgbm_rows = lgbm_rows.set_index("imbalance")[["macro_f1", "accuracy", "train_time_s"]]

lgbm_recall = pd.DataFrame({
    imb: recall_score(y_test_multi, multi_models[("LightGBM", imb)].predict(X_test),
                      labels=classes, average=None)
    for imb in ["none", "balanced", "smote"]
}, index=[f"recall_{c}" for c in classes]).T

lgbm_rows.join(lgbm_recall).round(3)

,macro_f1,accuracy,train_time_s,recall_Benign,recall_Bot,recall_DDoS,recall_Dos,recall_Prob
imbalance,,,,,,,,
none,0.647,0.882,28.26,0.924,0.055,0.835,0.907,0.475
balanced,0.615,0.829,29.17,0.873,0.871,0.487,0.887,0.709
smote,0.622,0.832,107.51,0.874,0.859,0.515,0.886,0.709


## 6. Hyperparameter tuning
We tune the multi-class Random Forest and LightGBM, because the best model is chosen on the multi-class task and this is where the hard classes (Bot, DDoS, Prob) are.

- **Search data:** a stratified 20% sample of the training set (about 68k rows), to keep the search time reasonable. The test set is not used.
- **RandomizedSearchCV:** tries 20 random combinations instead of the full grid.
- **Cross-validation:** `StratifiedKFold(n_splits=3, shuffle=True)`. Each combination is trained on 2 folds and scored on the third, three times, and the scores are averaged.
- **Score:** `f1_macro`, the main metric.
- **class_weight** is part of the search (`None` or `"balanced"`), so cross-validation decides the trade-off seen in section 5.
- Each model uses one core inside the search and the search runs 16 fits in parallel (`n_jobs=-1`). `refit=False` because we retrain on the full training set ourselves.

After the search, the best parameters are used to retrain on the **full training set**, and the model is measured once on the test set with `evaluate()`.

In [21]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train_multi, train_size=0.2, stratify=y_train_multi, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print('Tuning sample:', X_tune.shape)
print((y_tune.value_counts(normalize=True) * 100).round(2))

Tuning sample: (68326, 62)
label_multi
Benign    54.74
Dos       29.62
DDoS      11.77
Bot        2.10
Prob       1.77
Name: proportion, dtype: float64


### Random Forest search

In [22]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "class_weight": [None, "balanced"],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    rf_param_dist, n_iter=20, scoring="f1_macro", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, refit=False, verbose=1,
)

t0 = time.perf_counter()
rf_search.fit(X_tune, y_tune)
print(f"Search time: {(time.perf_counter() - t0) / 60:.1f} min")
print("Best CV macro F1:", round(rf_search.best_score_, 4))
print("Best params:", rf_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


Search time: 27.2 min
Best CV macro F1: 0.6559
Best params: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 30, 'class_weight': None}


### LightGBM search
`subsample` only has an effect in LightGBM when `subsample_freq` > 0, so `subsample_freq=1` is set.

In [23]:
lgbm_param_dist = {
    "n_estimators": [100, 200, 300, 500, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 10, 20],
    "min_child_samples": [10, 20, 50, 100],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "class_weight": [None, "balanced"],
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(random_state=RANDOM_STATE, n_jobs=1, verbose=-1, subsample_freq=1),
    lgbm_param_dist, n_iter=20, scoring="f1_macro", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, refit=False, verbose=1,
)

t0 = time.perf_counter()
lgbm_search.fit(X_tune, y_tune)
print(f"Search time: {(time.perf_counter() - t0) / 60:.1f} min")
print("Best CV macro F1:", round(lgbm_search.best_score_, 4))
print("Best params:", lgbm_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


Search time: 4.3 min
Best CV macro F1: 0.652
Best params: {'subsample': 0.6, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 50, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8, 'class_weight': None}


Top 5 combinations of each search (mean and standard deviation of the 3 CV scores).

In [24]:
def top_results(search, n=5):
    df = pd.DataFrame(search.cv_results_)
    cols = ["params", "mean_test_score", "std_test_score", "mean_fit_time"]
    return df.sort_values("rank_test_score")[cols].head(n).round(4)

pd.set_option("display.max_colwidth", None)
display(top_results(rf_search))
display(top_results(lgbm_search))

,params,mean_test_score,std_test_score,mean_fit_time
11,"{'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 30, 'class_weight': None}",0.6559,0.0066,396.8655
1,"{'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30, 'class_weight': 'balanced'}",0.6556,0.0041,185.3409
14,"{'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': None}",0.6551,0.0058,255.6070
13,"{'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': None, 'class_weight': None}",0.6539,0.0073,916.0195
4,"{'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': None, 'class_weight': None}",0.6530,0.0072,639.1011


,params,mean_test_score,std_test_score,mean_fit_time
17,"{'subsample': 0.6, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 50, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8, 'class_weight': None}",0.6520,0.0008,26.7724
14,"{'subsample': 1.0, 'num_leaves': 127, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 20, 'learning_rate': 0.03, 'colsample_bytree': 1.0, 'class_weight': None}",0.6504,0.0026,35.5614
5,"{'subsample': 0.6, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 100, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'class_weight': None}",0.6496,0.0059,47.5433
19,"{'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 1000, 'min_child_samples': 10, 'max_depth': 20, 'learning_rate': 0.01, 'colsample_bytree': 0.6, 'class_weight': None}",0.6463,0.0063,30.2414
8,"{'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 10, 'max_depth': 10, 'learning_rate': 0.03, 'colsample_bytree': 0.8, 'class_weight': None}",0.6447,0.0060,53.6026


### Retrain with the best parameters on the full training set

In [25]:
def tuned_row(row, best_params):
    row["imbalance"] = "balanced" if best_params.get("class_weight") == "balanced" else "none"
    params = {k: v for k, v in best_params.items() if k != "class_weight"}
    row["notes"] = f"tuned {params}"
    return row

rf_tuned = RandomForestClassifier(**rf_search.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
row = evaluate(rf_tuned, "RandomForest", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
results.append(tuned_row(row, rf_search.best_params_))
row

=== RandomForest | multi ===


              precision    recall  f1-score   support

      Benign     0.9324    0.9236    0.9280     46749
         Bot     0.3469    0.2372    0.2817      1792
        DDoS     0.5791    0.7040    0.6355     10054
         Dos     0.9362    0.9097    0.9228     25301
        Prob     0.7220    0.5116    0.5988      1513

    accuracy                         0.8719     85409
   macro avg     0.7033    0.6572    0.6734     85409
weighted avg     0.8759    0.8719    0.8726     85409



{'model': 'RandomForest',
 'task': 'multi',
 'imbalance': 'none',
 'macro_f1': 0.6733619187076945,
 'accuracy': 0.8719221627697316,
 'train_time_s': 77.72,
 'predict_ms_per_flow': 0.00599,
 'notes': "tuned {'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 30}"}

In [26]:
lgbm_tuned = LGBMClassifier(**lgbm_search.best_params_, subsample_freq=1,
                            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
row = evaluate(lgbm_tuned, "LightGBM", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
results.append(tuned_row(row, lgbm_search.best_params_))
row

=== LightGBM | multi ===


              precision    recall  f1-score   support

      Benign     0.9452    0.9241    0.9345     46749
         Bot     0.4113    0.1138    0.1783      1792
        DDoS     0.5781    0.8061    0.6734     10054
         Dos     0.9520    0.9105    0.9308     25301
        Prob     0.7432    0.4858    0.5875      1513

    accuracy                         0.8814     85409
   macro avg     0.7260    0.6481    0.6609     85409
weighted avg     0.8892    0.8814    0.8807     85409



{'model': 'LightGBM',
 'task': 'multi',
 'imbalance': 'none',
 'macro_f1': 0.66090766986114,
 'accuracy': 0.8814293575618495,
 'train_time_s': 34.74,
 'predict_ms_per_flow': 0.02406,
 'notes': "tuned {'subsample': 0.6, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 50, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8}"}